In [5]:
from dotenv import load_dotenv
import os
import requests


load_dotenv()

api_key = os.getenv("TMDB_API_KEY")


### Fetching data from tmdb api endpoint

In [6]:
# ── Handling pagination + rate limits in one reusable function ──────────────

import time


def fetch_all_pages(endpoint, headers=None, params=None, max_pages=10, delay=1.0):
    """
    📋 COPY-PASTE TEMPLATE
    Many APIs return data in pages with a "next" cursor/URL in the response.
    This loop keeps calling until there's no more "next" page or max_pages
    is hit (a safety cap so a bug can't loop forever).
    """
    all_results = []
    url = endpoint
    params = dict(params or {})

    for page in range(1,max_pages+1):
        params["page"]=page
        
        response = requests.get(url, headers=headers, params=params, timeout=10)

        if response.status_code == 429:
            # 429 = "Too Many Requests" — the API is rate-limiting you.
            # Respect the Retry-After header if present, else back off a fixed amount.
            wait = int(response.headers.get("Retry-After", 5))
            print(f"Rate limited, sleeping {wait}s...")
            time.sleep(wait)
            continue   # retry the SAME request after waiting

        response.raise_for_status()
        payload = response.json()

        all_results.extend(payload.get("results", []))

        print(f"Fetched page {page}/{max_pages}")
        time.sleep(delay)

    print("all pages fetched successfully")
    return all_results






In [7]:
url = "https://api.themoviedb.org/3/movie/top_rated"

params = {
    "api_key": api_key,
    "language": "en-US"
}
df_data = fetch_all_pages(
    url,
    params=params,
    max_pages=50
)

Fetched page 1/50
Fetched page 2/50
Fetched page 3/50
Fetched page 4/50
Fetched page 5/50
Fetched page 6/50
Fetched page 7/50
Fetched page 8/50
Fetched page 9/50
Fetched page 10/50
Fetched page 11/50
Fetched page 12/50
Fetched page 13/50
Fetched page 14/50
Fetched page 15/50
Fetched page 16/50
Fetched page 17/50
Fetched page 18/50
Fetched page 19/50
Fetched page 20/50
Fetched page 21/50
Fetched page 22/50
Fetched page 23/50
Fetched page 24/50
Fetched page 25/50
Fetched page 26/50
Fetched page 27/50
Fetched page 28/50
Fetched page 29/50
Fetched page 30/50
Fetched page 31/50
Fetched page 32/50
Fetched page 33/50
Fetched page 34/50
Fetched page 35/50
Fetched page 36/50
Fetched page 37/50
Fetched page 38/50
Fetched page 39/50
Fetched page 40/50
Fetched page 41/50
Fetched page 42/50
Fetched page 43/50
Fetched page 44/50
Fetched page 45/50
Fetched page 46/50
Fetched page 47/50
Fetched page 48/50
Fetched page 49/50
Fetched page 50/50
all pages fetched successfully


### Fetching data from tmdb single genre id and name page

In [8]:

import os



API_KEY = os.environ.get("MY_API_KEY", "")   # set this in your shell/`.env` file, not here

def fetch_from_authenticated_api(endpoint, params=None):
    """
    📋 COPY-PASTE TEMPLATE for a typical bearer-token REST API call.
    """
    headers = {"Authorization": f"Bearer {API_KEY}"}
    response = requests.get(endpoint, headers=headers, params=params, timeout=10)
    response.raise_for_status()
    return response.json()


df_genre=fetch_from_authenticated_api(f"https://api.themoviedb.org/3/genre/movie/list?api_key={api_key}&language=en-US")
genres=df_genre["genres"]



### converting into pandas dataframe

In [9]:
import pandas as pd

movie_df=pd.DataFrame(df_data)
genre_df=pd.DataFrame(genres)

### joining both dataframe on basis of genre id and name

In [10]:
movie_df=movie_df.explode("genre_ids")

df=movie_df.merge(genre_df,left_on="genre_ids", right_on="id")


df=df.drop(columns=["backdrop_path","poster_path","id_y","genre_ids"])

df = df.groupby("id_x").agg(
    lambda x: ", ".join(x.astype(str)) if x.name == "name" else x.iloc[0]
).reset_index()

df=df.rename(columns={"id_x":"id", "name":"genre"})

df.head()

,id,adult,title,original_language,original_title,overview,popularity,release_date,softcore,video,vote_average,vote_count,genre
0,11,False,Star Wars,en,Star Wars,Princess Leia is captured and held hostage by ...,31.8560,1977-05-25,False,False,8.207,22831,"Adventure, Action, Science Fiction"
1,12,False,Finding Nemo,en,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...",31.3140,2003-05-30,False,False,7.821,20948,"Animation, Family, Adventure"
2,13,False,Forrest Gump,en,Forrest Gump,A man with a low IQ has accomplished great thi...,33.6938,1994-06-23,False,False,8.462,30413,"Comedy, Drama, Romance"
3,14,False,American Beauty,en,American Beauty,"Lester Burnham, a depressed suburban father in...",16.9277,1999-09-15,False,False,7.996,13323,Drama
4,15,False,Citizen Kane,en,Citizen Kane,Newspaper magnate Charles Foster Kane is taken...,10.9652,1941-04-17,False,False,7.967,6213,"Mystery, Drama"


In [11]:
df["id"].dtype

dtype('int64')

### Data cleaning

In [12]:

SLANG_DICT = {
    # Common short forms
    "u": "you",
    "ur": "your",
    "r": "are",
    "ya": "you",
    "yr": "your",
    "n": "and",
    "b4": "before",
    "2": "to",
    "4": "for",

    # Common abbreviations
    "lol": "laughing out loud",
    "lmao": "laughing my ass off",
    "lmfao": "laughing my fucking ass off",
    "rofl": "rolling on the floor laughing",
    "omg": "oh my god",
    "omw": "on my way",
    "idk": "i do not know",
    "idc": "i do not care",
    "ik": "i know",
    "ikr": "i know right",
    "imo": "in my opinion",
    "imho": "in my humble opinion",
    "btw": "by the way",
    "brb": "be right back",
    "bbl": "be back later",
    "ttyl": "talk to you later",
    "ttys": "talk to you soon",
    "afk": "away from keyboard",

    # Agreement / reaction
    "yep": "yes",
    "yup": "yes",
    "nope": "no",
    "yeah": "yes",
    "nah": "no",
    "fr": "for real",
    "frfr": "for real for real",
    "ngl": "not gonna lie",
    "tbh": "to be honest",
    "honestly": "honestly",
    "istg": "i swear to god",
    "smh": "shaking my head",
    "fyi": "for your information",
    "afaik": "as far as i know",

    # Time / urgency
    "asap": "as soon as possible",
    "rn": "right now",
    "atm": "at the moment",
    "l8r": "later",
    "cya": "see you",
    "g2g": "got to go",
    "gtg": "got to go",

    # Thanks / requests
    "thx": "thanks",
    "thanx": "thanks",
    "ty": "thank you",
    "tysm": "thank you so much",
    "yw": "you are welcome",
    "pls": "please",
    "plz": "please",
    "np": "no problem",
    "nvm": "never mind",

    # Emotions / reactions
    "wtf": "what the fuck",
    "wth": "what the hell",
    "tf": "the fuck",
    "ffs": "for fucks sake",
    "omfg": "oh my fucking god",
    "yay": "yay",
    "ugh": "ugh",
    "aww": "aww",

    # Positive / negative
    "gr8": "great",
    "def": "definitely",
    "lit": "exciting",
    "fire": "excellent",
    "goat": "greatest of all time",
    "w": "win",
    "l": "loss",
    "sus": "suspicious",
    "mid": "average",
    "meh": "not very good",

    # Social media / internet
    "dm": "direct message",
    "pm": "private message",
    "irl": "in real life",
    "iirc": "if i remember correctly",
    "tmi": "too much information",
    "fomo": "fear of missing out",
    "yolo": "you only live once",
    "bff": "best friend forever",
    "bffl": "best friends for life",

    # Relationships / people
    "bf": "boyfriend",
    "gf": "girlfriend",
    "bday": "birthday",
    "hbd": "happy birthday",
    "fam": "family",
    "bro": "brother",
    "sis": "sister",

    # Common conversational phrases
    "wyd": "what are you doing",
    "wbu": "what about you",
    "hbu": "how about you",
    "wym": "what do you mean",
    "wdym": "what do you mean",
    "hmu": "hit me up",
    "lmk": "let me know",
    "msg": "message",
    "cmon": "come on",
    "gonna": "going to",
    "wanna": "want to",
    "gotta": "got to",
    "kinda": "kind of",
    "sorta": "sort of",

    # Your original examples
    "soooo": "so",
}

In [13]:
import re
import string
import contractions
import emoji
from spellchecker import SpellChecker
import unidecode


def to_lowercase(text: str) -> str:
    return text.lower()

def remove_mentions(text: str) -> str:
    return re.sub(r"@\w+", "", text)

def strip_hashtag_symbol(text: str) -> str:
    # keeps the word, removes only the '#'
    return re.sub(r"#(\w+)", r"\1", text)

def remove_accented_chars(text: str) -> str:
    return unidecode.unidecode(text)

def emojis_to_text(text: str) -> str:
    # 😍 -> :heart_eyes: -> heart eyes
    demojized = emoji.demojize(text, delimiters=(" ", " "))
    return demojized.replace("_", " ").replace(":", "")

def expand_contractions(text: str) -> str:
    return contractions.fix(text)

def expand_slang(text: str, slang_dict: dict = SLANG_DICT) -> str:
    words = text.split()
    expanded = [slang_dict.get(w.lower(), w) for w in words]
    return " ".join(expanded)

def remove_punctuation(text: str) -> str:
    return text.translate(str.maketrans("", "", string.punctuation))

def remove_extra_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

spell = SpellChecker()

def correct_spelling(text: str) -> str:
    words = text.split()
    corrected = []
    for w in words:
        fix = spell.correction(w)
        corrected.append(fix if fix else w)
    return " ".join(corrected)



In [14]:
def preprocess(text):
    text = to_lowercase(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = expand_slang(text)
    text = strip_hashtag_symbol(text)
    text = remove_accented_chars(text)
    text = emojis_to_text(text)
    text = expand_contractions(text)
    text = correct_spelling(text)
    text = remove_extra_whitespace(text)

    return text

In [15]:
columns=['id', 'adult', 'title', 'original_language', 'original_title',
       'overview', 'popularity', 'release_date', 'softcore', 'video',
       'vote_average', 'vote_count', 'genre']

for col in columns:
    if df[col].dtype == "string":
        df[col]=df[col].apply(preprocess)
    

In [16]:
df.head()

,id,adult,title,original_language,original_title,overview,popularity,release_date,softcore,video,vote_average,vote_count,genre
0,11,False,star wars,en,star wars,princess lei is captured and held hostage by t...,31.8560,19770525,False,False,8.207,22831,adventure action science fiction
1,12,False,finding memo,en,finding memo,memo an adventurous young clownish is unexpect...,31.3140,20030530,False,False,7.821,20948,animation family adventure
2,13,False,forrest jump,en,forrest jump,a man with a low i has accomplished great thin...,33.6938,19940623,False,False,8.462,30413,comedy drama romance
3,14,False,american beauty,en,american beauty,lester durham a depressed suburban father in a...,16.9277,19990915,False,False,7.996,13323,drama
4,15,False,citizen kane,en,citizen kane,newspaper magnate charles foster kane is taken...,10.9652,19410417,False,False,7.967,6213,mystery drama
